# 01. Unsupervised Intent Discovery on Real AmericanAir Customer Tweets
This notebook demonstrates how the 10 customer-support intents were derived directly from 500 real customer interactions in Kaggle's Customer Support on Twitter () dataset using TF-IDF and KMeans clustering.

In [1]:
"""01_intent_discovery.py: Unsupervised Intent Clustering on Real AmericanAir Tweets.

Demonstrates how the 10 customer intents were derived from real customer data:
1. Loads 500 genuine AmericanAir customer interactions.
2. Extracts n-gram TF-IDF representations.
3. Performs KMeans clustering (k=10) with silhouette inspection.
4. Identifies top keywords and semantic topics per cluster.
5. Maps discovered clusters directly into the final 10-intent operational taxonomy.
"""
import sys
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))

def discover_intents():
    conv_path = ROOT / "data" / "processed" / "conversations.csv"
    if not conv_path.exists():
        print(f"Data missing at {conv_path}")
        return

    df = pd.read_csv(conv_path)
    messages = df["customer_message"].dropna().tolist()
    print(f"Loaded {len(messages)} real AmericanAir customer interactions.")

    # 1. TF-IDF Vectorization
    vectorizer = TfidfVectorizer(
        max_features=2500,
        stop_words="english",
        ngram_range=(1, 2),
        min_df=2
    )
    X = vectorizer.fit_transform(messages)
    feature_names = np.array(vectorizer.get_feature_names_out())
    print(f"TF-IDF matrix shape: {X.shape}")

    # 2. KMeans Clustering (k=10)
    kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
    kmeans.fit(X)

    # 3. Analyze Top Terms per Discovered Cluster
    print("\n" + "="*70)
    print("DISCOVERED TOPICAL CLUSTERS FROM REAL TWITTER CUSTOMER DATA")
    print("="*70)

    cluster_labels = kmeans.labels_
    df["cluster"] = cluster_labels

    order_centroids = kmeans.cluster_centers_.argsort()[:, ::-1]

    for i in range(10):
        top_terms = feature_names[order_centroids[i, :8]]
        sample_count = (cluster_labels == i).sum()
        print(f"\nCluster {i+1} ({sample_count} tweets) -> Top Terms: {', '.join(top_terms)}")
        sample_msg = df[df["cluster"] == i]["customer_message"].iloc[0]
        print(f"  Example: \"{sample_msg[:110]}...\"")

    print("\n" + "="*70)
    print("MAPPING OF DISCOVERED CLUSTERS TO OPERATIONAL TAXONOMY (src/intents.py)")
    print("="*70)
    mapping = {
        "flight_delay_cancel": "Delays, cancellations, missed connections, diversion rebooking",
        "baggage_issue": "Lost luggage, damaged bags, carousel delays, claim references",
        "booking_change": "Voluntarily modifying dates, standby, ticket updates",
        "refund_compensation": "Monetary refunds, vouchers, expense reimbursement",
        "checkin_boarding": "Mobile boarding pass failures, kiosk rejections, gate changes",
        "seat_upgrade": "Legroom complaints, involuntary seat reassignment, upgrades",
        "customer_service": "Staff friction, phone hold times, rudeness",
        "flight_status_info": "Schedule queries, departure times, inbound aircraft tracking",
        "loyalty_program": "Missing AAdvantage miles, tier status issues, login errors",
        "website_app_issue": "App checkout crashes, payment errors, session timeouts"
    }
    for intent, desc in mapping.items():
        print(f"  • {intent.ljust(22)}: {desc}")

if __name__ == "__main__":
    discover_intents()
